In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open by name (or use open_by_url / open_by_key if you have the sheet's URL/ID)
sheet = gc.open('Capstone Copy of Appointments').worksheet('Appointments_2025/2026')
tenure_sheet= gc.open('Capstone Copy of Appointments').worksheet('Clinician_Tenure')

# Pull all data into a DataFrame
import pandas as pd
import json
data = sheet.get_all_records()
raw_df = pd.DataFrame(data)

tenure = tenure_sheet.get_all_records()
tenure_df = pd.DataFrame(tenure)


In [ ]:
# Mount Drive first — everything below depends on this
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
Github_Token = userdata.get('Github_Token')
!git remote set-url origin https://{Github_Token}@github.com/Sharion2023/Data_Analytics_Capstone.git

# Move into the repo (already cloned, lives permanently in Drive)
%cd /content/drive/MyDrive/Data_Analytics_Capstone

# Git identity (session-only, still needs to be set each time)
!git config user.name "Sharion2023"
!git config user.email "your-email@example.com"

# Activate nbstripout locally each session
!pip install nbstripout --quiet
!nbstripout --install

# Load staff_anon.json from Drive
import json
with open('/content/drive/MyDrive/staff_anon.json', 'r') as f:
    staff_anon = json.load(f)

print("✅ Drive mounted, repo location set, git configured, nbstripout active, staff_anon loaded.")

In [ ]:
#create df for calculated results
si_ratio_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Practitioner')
si_data = si_ratio_sheet.get_all_records()
si_df = pd.DataFrame(si_data)

In [ ]:
# Strip whitespace on the join key in both DataFrames
raw_df['staff_member_name'] = raw_df['staff_member_name'].str.strip()
tenure_df['staff_member_name'] = tenure_df['staff_member_name'].str.strip()
si_df['staff_member_name'] = si_df['staff_member_name'].str.strip()

# Merge
df = raw_df.merge(tenure_df, on='staff_member_name', how='left')

print(f"raw_df rows: {len(raw_df)} | merged df rows: {len(df)}")

In [ ]:
unmatched_count = df[df['start_date'].isna()]['staff_member_name'].nunique()
print(f"{unmatched_count} unique staff members did not match")

In [ ]:
df = df.merge(si_df, on=['staff_member_name', 'iso_week'], how='left')

In [ ]:
#map to aonymous mapping
df['clinician_id'] = df['staff_member_name'].map(staff_anon)

In [ ]:
#check that all mapping worked
unmatched = df['clinician_id'].isna().sum()
print(f"{unmatched} rows failed to map to a clinician_id")

In [ ]:
df.columns

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.columns

In [ ]:
#drop unnecessary columns
col_to_keep = [
    'patient_number',    # this is your patient ID field, not 'patient_id'
    'clinician_id',
    'start_at',
    'arrived_at',
    'first_visit',
    'treatment_name',
    'booked_at', # lead time analysis for intake-conversion stream
    'start_date',
    'iso_week',
    'subsequent_visits',
    'initial_visits',
    'real_date',
       'weekly_S/I_ratio',
    'rolling_4-week_S/I_ratio',
    'tenure_status',
       'unique_patients',
    'fall_off_patients',
    '4_wk_fall_off_patient_calc'
]

df_clean = df_clean[col_to_keep]

In [ ]:
df_clean.head()
df_clean.isna().sum()


In [ ]:
df_clean.head(10)

In [ ]:
df_clean.shape

In [ ]:
df_clean['clinician_id'].unique()

In [ ]:
# Confirm unique patient count, expect 4309
print(df_clean['patient_number'].nunique())

In [ ]:
!git status

In [ ]:
!git add 'SI_Ratio_Analysis.ipynb'
!git commit -m 'Data sources merged and anonymized'
!git push

In [ ]:
clinic_wide_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Clinic')
clinic_wide_data = clinic_wide_sheet.get_all_records()
clinic_wide_df = pd.DataFrame(clinic_wide_data)

In [ ]:
import matplotlib.pyplot as plt

#Calculate clinic wide S/I ratio

clinic_wide_df['rolling_4-week_S/I_ratio'] = pd.to_numeric(
    clinic_wide_df['rolling_4-week_S/I_ratio'], errors='coerce'
)
clinic_trend = clinic_wide_df.set_index('iso_week')['rolling_4-week_S/I_ratio']

clinic_trend.plot()
plt.title('Clinic S/I Ratio Trend')
plt.xlabel('Week')
plt.ylabel('S/I Ratio')
plt.xticks(rotation=45)
plt.show()

In [ ]:
!g

In [ ]:
summary = pd.DataFrame({
    'patient_count': visits_per_patient_by_clinician.groupby('staff_member_name').size(),
    'retention_rate': visits_per_patient_by_clinician.groupby('staff_member_name').apply(retention_rate)
})
print(summary.sort_values('retention_rate', ascending=False))

The variation in patient count is vast. I'll remove outliers for a cleaner interpretation.

In [ ]:
Q1 = summary['patient_count'].quantile(0.25)
Q3 = summary['patient_count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = summary[(summary['patient_count'] < lower_bound) | (summary['patient_count'] > upper_bound)]
print(f"Bounds: {lower_bound:.0f} to {upper_bound:.0f}")
print(outliers)

IQR method not effective on such a small data set. Will attempt

In [ ]:
MIN_CASELOAD = 50  # justify this number in your methodology write-up

reliable_summary = summary[summary['patient_count'] >= MIN_CASELOAD]
excluded = summary[summary['patient_count'] < MIN_CASELOAD]

print(f"Included: {len(reliable_summary)} clinicians")
print(f"Excluded (caseload < {MIN_CASELOAD}): {len(excluded)} clinicians")
print(excluded)

In [ ]:
from scipy import stats

# --- Full dataset (all 16 clinicians) ---
corr_full, p_full = stats.pearsonr(summary['patient_count'], summary['retention_rate'])
print("=== Full dataset (all clinicians) ===")
print(f"n = {len(summary)}")
print(f"Pearson r = {corr_full:.3f}, p = {p_full:.3f}")

print()

# --- Thresholded dataset (caseload >= MIN_CASELOAD) ---
corr_thresh, p_thresh = stats.pearsonr(reliable_summary['patient_count'], reliable_summary['retention_rate'])
print(f"=== Thresholded dataset (caseload >= {MIN_CASELOAD}) ===")
print(f"n = {len(reliable_summary)}")
print(f"Pearson r = {corr_thresh:.3f}, p = {p_thresh:.3f}")